<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 36px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  <b>00. Introduction to MCMC</b>
</div>

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">📑 Table of Contents</span>

- **1. [What is MCMC?](#-part-i-what-is-mcmc)**
    - 1.1 [The Sampling Problem](#the-sampling-problem)
    - 1.2 [Why Not Just Use Grid Search?](#why-not-just-use-grid-search)

- **2. [Setup & Imports](#-part-ii-setup--imports)**

- **3. [Metropolis-Hastings Algorithm](#-part-iii-metropolis-hastings-algorithm)**
    - 3.1 [The Algorithm Step by Step](#the-algorithm-step-by-step)
    - 3.2 [Implementation from Scratch](#implementation-from-scratch)
    - 3.3 [Visualizing the Markov Chain](#visualizing-the-markov-chain)

- **4. [MCMC with TensorFlow Probability](#-part-iv-mcmc-with-tensorflow-probability)**
    - 4.1 [Using tfp.mcmc.RandomWalkMetropolis](#using-tfpmcmcrandomwalkmetropolis)
    - 4.2 [Sampling from a Posterior Distribution](#sampling-from-a-posterior-distribution)

- **5. [Convergence Diagnostics](#-part-v-convergence-diagnostics)**
    - 5.1 [Trace Plots](#trace-plots)
    - 5.2 [Effective Sample Size](#effective-sample-size)
    - 5.3 [R-hat Diagnostic](#r-hat-diagnostic)

- **6. [Practical Bayesian Inference with MCMC](#-part-vi-practical-bayesian-inference)**
    - 6.1 [Bayesian Linear Regression Example](#bayesian-linear-regression-example)
    - 6.2 [Posterior Predictive Checks](#posterior-predictive-checks)

- **7. [Key Takeaways & Next Steps](#-part-vii-key-takeaways--next-steps)**

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">1. What is MCMC?</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">1.1. The Sampling Problem</span>

- **Markov Chain Monte Carlo (MCMC)** is a class of algorithms for sampling from probability distributions that are difficult to sample from directly.

- In Bayesian inference, we often need to compute the **posterior distribution**:

$$p(\theta | \mathcal{D}) = \frac{p(\mathcal{D} | \theta) \cdot p(\theta)}{p(\mathcal{D})}$$

- The denominator $p(\mathcal{D}) = \int p(\mathcal{D} | \theta) p(\theta) d\theta$ (the **marginal likelihood**) is usually **intractable** — it requires integrating over all possible parameter values.

- **Key Insight**: MCMC lets us draw samples from the posterior **without computing the normalizing constant**. We only need the unnormalized posterior $p(\mathcal{D} | \theta) \cdot p(\theta)$.

- **Applications in practice**:
    - Bayesian parameter estimation
    - Model comparison
    - Uncertainty quantification
    - Hierarchical modeling

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">1.2. Why Not Just Use Grid Search?</span>

| **Method** | **Dimensions** | **Scalability** | **Accuracy** |
|-----------|---------------|-----------------|-------------|
| Grid Search | 1–3 | ❌ Exponential cost | High (if fine grid) |
| Rejection Sampling | 1–5 | ❌ Low acceptance rate | Exact |
| Importance Sampling | 1–10 | ⚠️ Variance issues | Depends on proposal |
| **MCMC** | **1–1000+** | **✅ Scales well** | **Asymptotically exact** |
| Variational Inference | 1–millions | ✅ Very scalable | Approximate |

- MCMC provides the **gold standard** for Bayesian inference — it gives asymptotically exact samples.
- The **curse of dimensionality** makes grid-based methods impractical beyond a few dimensions.

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">2. Setup & Imports</span>

In [ ]:
# Core imports
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm

# Standard TFP aliases
tfd = tfp.distributions
tfb = tfp.bijectors

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Plot styling
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print(f"TensorFlow version: {tf.__version__}")
print(f"TensorFlow Probability version: {tfp.__version__}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">3. Metropolis-Hastings Algorithm</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.1. The Algorithm Step by Step</span>

The **Metropolis-Hastings algorithm** works as follows:

1. **Initialize**: Start at some point $\theta_0$
2. **Propose**: Draw a candidate $\theta^* \sim q(\theta^* | \theta_t)$ from a proposal distribution
3. **Compute acceptance ratio**:
$$\alpha = \min\left(1, \frac{p(\theta^*) \cdot q(\theta_t | \theta^*)}{p(\theta_t) \cdot q(\theta^* | \theta_t)}\right)$$
4. **Accept/Reject**: With probability $\alpha$, set $\theta_{t+1} = \theta^*$; otherwise $\theta_{t+1} = \theta_t$
5. **Repeat** steps 2–4

For a **symmetric** proposal distribution (e.g., Gaussian random walk), $q(\theta^* | \theta_t) = q(\theta_t | \theta^*)$, so the ratio simplifies to:
$$\alpha = \min\left(1, \frac{p(\theta^*)}{p(\theta_t)}\right)$$

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.2. Implementation from Scratch</span>

In [ ]:
def metropolis_hastings(target_log_prob_fn, initial_state, num_samples, step_size=0.5):
    """
    Simple Metropolis-Hastings sampler with Gaussian random walk proposal.
    
    Args:
        target_log_prob_fn: Function that returns log probability of the target
        initial_state: Starting point for the chain
        num_samples: Number of samples to draw
        step_size: Standard deviation of the proposal distribution
    
    Returns:
        samples: Array of MCMC samples
        acceptance_rate: Fraction of accepted proposals
    """
    samples = np.zeros((num_samples, len(initial_state)))
    current_state = np.array(initial_state, dtype=np.float64)
    current_log_prob = target_log_prob_fn(current_state)
    accepted = 0
    
    for i in range(num_samples):
        # Step 1: Propose a new state
        proposal = current_state + np.random.normal(0, step_size, size=len(current_state))
        proposal_log_prob = target_log_prob_fn(proposal)
        
        # Step 2: Compute acceptance probability (in log space)
        log_alpha = proposal_log_prob - current_log_prob
        
        # Step 3: Accept or reject
        if np.log(np.random.uniform()) < log_alpha:
            current_state = proposal
            current_log_prob = proposal_log_prob
            accepted += 1
        
        samples[i] = current_state
    
    acceptance_rate = accepted / num_samples
    return samples, acceptance_rate


# Target: A bimodal distribution (mixture of two Gaussians)
def bimodal_log_prob(x):
    """Log probability of a 1D mixture of two Gaussians."""
    component1 = -0.5 * ((x[0] - 3.0) / 1.0) ** 2
    component2 = -0.5 * ((x[0] + 3.0) / 1.5) ** 2
    return float(np.logaddexp(component1 + np.log(0.4), component2 + np.log(0.6)))


# Run the sampler
samples_mh, acc_rate = metropolis_hastings(
    target_log_prob_fn=bimodal_log_prob,
    initial_state=[0.0],
    num_samples=10000,
    step_size=1.5
)

print(f"Acceptance rate: {acc_rate:.3f}")
print(f"Sample mean: {samples_mh[:, 0].mean():.3f}")
print(f"Sample std: {samples_mh[:, 0].std():.3f}")

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.3. Visualizing the Markov Chain</span>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Trace plot
axes[0].plot(samples_mh[:2000, 0], alpha=0.7, linewidth=0.5, color='b')
axes[0].set_xlabel('Iteration', fontsize=13)
axes[0].set_ylabel('Sample Value', fontsize=13)
axes[0].set_title('Trace Plot (first 2000 samples)', fontsize=15, fontweight='bold')
axes[0].axhline(y=3.0, color='r', linestyle='--', alpha=0.5, label='Mode 1')
axes[0].axhline(y=-3.0, color='g', linestyle='--', alpha=0.5, label='Mode 2')
axes[0].legend()

# Histogram vs true density
x_range = np.linspace(-10, 10, 500)
true_density = 0.4 * np.exp(-0.5 * ((x_range - 3.0) / 1.0)**2) / (1.0 * np.sqrt(2*np.pi)) + \
               0.6 * np.exp(-0.5 * ((x_range + 3.0) / 1.5)**2) / (1.5 * np.sqrt(2*np.pi))

burn_in = 1000  # Discard first 1000 samples
axes[1].hist(samples_mh[burn_in:, 0], bins=80, density=True, alpha=0.6, 
             color='m', label='MCMC samples')
axes[1].plot(x_range, true_density, 'r-', linewidth=2.5, label='True density')
axes[1].set_xlabel('x', fontsize=13)
axes[1].set_ylabel('Density', fontsize=13)
axes[1].set_title('MCMC Samples vs True Distribution', fontsize=15, fontweight='bold')
axes[1].legend(fontsize=12)

plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">4. MCMC with TensorFlow Probability</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">4.1. Using tfp.mcmc.RandomWalkMetropolis</span>

In [ ]:
# Define the target distribution: a 2D correlated Gaussian
target_dist = tfd.MultivariateNormalFullCovariance(
    loc=[1.0, 2.0],
    covariance_matrix=[[1.0, 0.8],
                       [0.8, 1.5]]
)

# Create the MCMC kernel
rwm_kernel = tfp.mcmc.RandomWalkMetropolis(
    target_log_prob_fn=target_dist.log_prob,
    new_state_fn=tfp.mcmc.random_walk_normal_fn(scale=0.5)
)

# Run the chain
@tf.function
def run_chain():
    samples, kernel_results = tfp.mcmc.sample_chain(
        num_results=5000,
        num_burnin_steps=1000,
        current_state=[0.0, 0.0],
        kernel=rwm_kernel,
        trace_fn=lambda _, pkr: pkr.is_accepted
    )
    return samples, kernel_results

samples_2d, is_accepted = run_chain()

print(f"Shape of samples: {samples_2d.shape}")
print(f"Acceptance rate: {tf.reduce_mean(tf.cast(is_accepted, tf.float32)):.3f}")
print(f"Sample mean: {tf.reduce_mean(samples_2d, axis=0).numpy()}")
print(f"True mean: {target_dist.mean().numpy()}")

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">4.2. Sampling from a Posterior Distribution</span>

In [ ]:
# Generate synthetic data from a simple linear model: y = 2x + 1 + noise
np.random.seed(42)
n_data = 50
x_data = np.random.uniform(-2, 2, n_data).astype(np.float32)
y_data = (2.0 * x_data + 1.0 + np.random.normal(0, 0.5, n_data)).astype(np.float32)

# Define the log posterior: log p(slope, intercept | data)
def target_log_prob_fn(slope, intercept):
    """Unnormalized log posterior for Bayesian linear regression."""
    # Prior: Normal(0, 10) for both parameters
    prior_slope = tfd.Normal(loc=0., scale=10.)
    prior_intercept = tfd.Normal(loc=0., scale=10.)
    
    # Likelihood: y ~ Normal(slope * x + intercept, sigma=0.5)
    predicted = slope * x_data + intercept
    likelihood = tfd.Normal(loc=predicted, scale=0.5)
    
    return (
        prior_slope.log_prob(slope) +
        prior_intercept.log_prob(intercept) +
        tf.reduce_sum(likelihood.log_prob(y_data))
    )

# MCMC sampling using Random Walk Metropolis
rwm_kernel = tfp.mcmc.RandomWalkMetropolis(
    target_log_prob_fn=target_log_prob_fn,
    new_state_fn=tfp.mcmc.random_walk_normal_fn(scale=0.1)
)

@tf.function
def run_posterior_chain():
    samples, kernel_results = tfp.mcmc.sample_chain(
        num_results=5000,
        num_burnin_steps=2000,
        current_state=[0.0, 0.0],  # [slope_init, intercept_init]
        kernel=rwm_kernel,
        trace_fn=lambda _, pkr: pkr.is_accepted
    )
    return samples, kernel_results

posterior_samples, accepted = run_posterior_chain()
slope_samples, intercept_samples = posterior_samples

print(f"Estimated slope:     {tf.reduce_mean(slope_samples):.3f} ± {tf.math.reduce_std(slope_samples):.3f}")
print(f"True slope:          2.000")
print(f"Estimated intercept: {tf.reduce_mean(intercept_samples):.3f} ± {tf.math.reduce_std(intercept_samples):.3f}")
print(f"True intercept:      1.000")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Joint posterior
axes[0].scatter(slope_samples.numpy(), intercept_samples.numpy(), 
                alpha=0.1, s=5, color='m')
axes[0].axvline(x=2.0, color='red', linestyle='--', label='True slope')
axes[0].axhline(y=1.0, color='red', linestyle='--', label='True intercept')
axes[0].set_xlabel('Slope', fontsize=13)
axes[0].set_ylabel('Intercept', fontsize=13)
axes[0].set_title('Joint Posterior', fontsize=15, fontweight='bold')
axes[0].legend()

# Marginal: slope
axes[1].hist(slope_samples.numpy(), bins=50, density=True, 
             alpha=0.7, color='b', label='Posterior')
axes[1].axvline(x=2.0, color='red', linewidth=2, linestyle='--', label='True value')
axes[1].set_xlabel('Slope', fontsize=13)
axes[1].set_title('Marginal Posterior: Slope', fontsize=15, fontweight='bold')
axes[1].legend()

# Marginal: intercept
axes[2].hist(intercept_samples.numpy(), bins=50, density=True, 
             alpha=0.7, color='g', label='Posterior')
axes[2].axvline(x=1.0, color='red', linewidth=2, linestyle='--', label='True value')
axes[2].set_xlabel('Intercept', fontsize=13)
axes[2].set_title('Marginal Posterior: Intercept', fontsize=15, fontweight='bold')
axes[2].legend()

plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">5. Convergence Diagnostics</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">5.1. Trace Plots</span>

Trace plots show the sampled values over iterations. A **well-mixed chain** looks like a "hairy caterpillar" — random fluctuations around the mean without trends or long-range correlations.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(slope_samples.numpy(), alpha=0.7, linewidth=0.5, color='b')
axes[0].set_ylabel('Slope', fontsize=13)
axes[0].set_title('Trace Plots', fontsize=15, fontweight='bold')
axes[0].axhline(y=2.0, color='red', linestyle='--', alpha=0.7)

axes[1].plot(intercept_samples.numpy(), alpha=0.7, linewidth=0.5, color='g')
axes[1].set_ylabel('Intercept', fontsize=13)
axes[1].set_xlabel('Iteration', fontsize=13)
axes[1].axhline(y=1.0, color='red', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">5.2. Effective Sample Size</span>

Due to autocorrelation in MCMC chains, not all samples are independent. The **Effective Sample Size (ESS)** tells us how many independent samples our chain is equivalent to.

In [ ]:
# Compute ESS using TFP's built-in function
ess_slope = tfp.mcmc.effective_sample_size(slope_samples)
ess_intercept = tfp.mcmc.effective_sample_size(intercept_samples)

print(f"Total samples: {len(slope_samples)}")
print(f"ESS (slope):     {ess_slope.numpy():.1f} ({ess_slope.numpy()/len(slope_samples)*100:.1f}%)")
print(f"ESS (intercept): {ess_intercept.numpy():.1f} ({ess_intercept.numpy()/len(intercept_samples)*100:.1f}%)")
print(f"\n💡 Rule of thumb: ESS > 400 is generally sufficient for reliable posterior estimates.")

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">5.3. R-hat Diagnostic</span>

The **R-hat** (Gelman-Rubin) diagnostic compares within-chain and between-chain variance. Values close to **1.0** indicate convergence. A common threshold is $\hat{R} < 1.1$.

In [ ]:
# Run multiple chains for R-hat computation
@tf.function
def run_multiple_chains(num_chains=4):
    initial_states = [
        tf.random.normal([num_chains]),  # slope
        tf.random.normal([num_chains])   # intercept
    ]
    
    samples, _ = tfp.mcmc.sample_chain(
        num_results=3000,
        num_burnin_steps=1000,
        current_state=initial_states,
        kernel=tfp.mcmc.RandomWalkMetropolis(
            target_log_prob_fn=target_log_prob_fn,
            new_state_fn=tfp.mcmc.random_walk_normal_fn(scale=0.1)
        ),
        trace_fn=lambda _, pkr: pkr.is_accepted
    )
    return samples

multi_chain_samples = run_multiple_chains()

# Compute R-hat
rhat_slope = tfp.mcmc.potential_scale_reduction(multi_chain_samples[0])
rhat_intercept = tfp.mcmc.potential_scale_reduction(multi_chain_samples[1])

print(f"R-hat (slope):     {rhat_slope.numpy():.4f}")
print(f"R-hat (intercept): {rhat_intercept.numpy():.4f}")
print(f"\n✅ Values close to 1.0 indicate convergence (threshold: < 1.1)")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">6. Practical Bayesian Inference with MCMC</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">6.1. Bayesian Linear Regression Example</span>

In [ ]:
# Posterior predictive: draw regression lines from posterior samples
fig, ax = plt.subplots(figsize=(12, 7))

# Plot data
ax.scatter(x_data, y_data, c='k', s=60, zorder=5, 
           edgecolors='white', linewidth=1, label='Observed data')

# Plot posterior regression lines (subset for clarity)
x_pred = np.linspace(-2.5, 2.5, 100)
n_lines = 100
indices = np.random.choice(len(slope_samples), n_lines, replace=False)

for i in indices:
    y_pred = slope_samples[i].numpy() * x_pred + intercept_samples[i].numpy()
    ax.plot(x_pred, y_pred, color='m', alpha=0.05, linewidth=1)

# Plot true line
ax.plot(x_pred, 2.0 * x_pred + 1.0, 'r--', linewidth=2.5, label='True: y = 2x + 1')

# Plot posterior mean line
mean_slope = tf.reduce_mean(slope_samples).numpy()
mean_intercept = tf.reduce_mean(intercept_samples).numpy()
ax.plot(x_pred, mean_slope * x_pred + mean_intercept, 
        color='c', linewidth=2.5, label=f'Posterior mean: y = {mean_slope:.2f}x + {mean_intercept:.2f}')

ax.set_xlabel('x', fontsize=14)
ax.set_ylabel('y', fontsize=14)
ax.set_title('Bayesian Linear Regression with MCMC\n(100 posterior samples shown)', 
             fontsize=16, fontweight='bold')
ax.legend(fontsize=12, loc='upper left')
plt.tight_layout()
plt.show()

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">6.2. Posterior Predictive Checks</span>

Posterior predictive checks help validate our model by comparing replicated data (generated from the posterior) with the observed data.

In [ ]:
# Generate posterior predictive samples
n_ppc = 200
ppc_indices = np.random.choice(len(slope_samples), n_ppc, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Posterior predictive distribution for a specific x value
x_test = 1.5
y_pred_samples = []
for i in ppc_indices:
    mean_pred = slope_samples[i].numpy() * x_test + intercept_samples[i].numpy()
    y_pred = np.random.normal(mean_pred, 0.5)  # Add observation noise
    y_pred_samples.append(y_pred)

axes[0].hist(y_pred_samples, bins=30, density=True, alpha=0.7, 
             color='m', edgecolor='white')
axes[0].axvline(x=2.0*x_test + 1.0, color='red', linewidth=2, 
                linestyle='--', label=f'True y at x={x_test}')
axes[0].set_xlabel('Predicted y', fontsize=13)
axes[0].set_ylabel('Density', fontsize=13)
axes[0].set_title(f'Posterior Predictive at x = {x_test}', fontsize=15, fontweight='bold')
axes[0].legend(fontsize=12)

# Residual check
residuals = []
for i in ppc_indices:
    pred = slope_samples[i].numpy() * x_data + intercept_samples[i].numpy()
    residuals.append(y_data - pred)
residuals = np.array(residuals)
mean_residuals = residuals.mean(axis=0)

axes[1].scatter(x_data, mean_residuals, c='b', s=50, alpha=0.7, edgecolors='white')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('x', fontsize=13)
axes[1].set_ylabel('Mean Residual', fontsize=13)
axes[1].set_title('Posterior Mean Residuals', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">7. Key Takeaways & Next Steps</span>

### Key Takeaways

| **Concept** | **Key Point** |
|------------|---------------|
| MCMC Idea | Sample from distributions you can't compute directly |
| Metropolis-Hastings | Accept/reject proposals based on probability ratio |
| Burn-in | Discard initial samples before chain converges |
| ESS | Effective number of independent samples (aim for > 400) |
| R-hat | Convergence diagnostic across chains (aim for < 1.1) |
| TFP Integration | `tfp.mcmc` provides production-ready MCMC tools |

### Next Steps

- **Notebook 01**: Learn about **Hamiltonian Monte Carlo (HMC)**, which dramatically improves sampling efficiency by using gradient information.
- **Notebook 02**: Explore **Bayesian Model Comparison** using marginal likelihoods and Bayes factors.
- **Notebook 03**: Build **Hierarchical Bayesian Models** with multi-level structure using `tfp.mcmc`.